<a href="https://colab.research.google.com/github/LinaMariaCastro/curso-ia-para-economia/blob/main/clases/3_Analisis_y_visualizacion_datos/2_Combinacion_Exportacion_Datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Inteligencia Artificial con Aplicaciones en Economía I**

- 👩‍🏫 **Profesora:** [Lina María Castro](https://www.linkedin.com/in/lina-maria-castro)  
- 📧 **Email:** [lmcastroco@gmail.com](mailto:lmcastroco@gmail.com)  
- 🎓 **Universidad:** Universidad Externado de Colombia - Facultad de Economía

# 🛢️**Métodos de combinación y exportación de datos**

**Objetivos de Aprendizaje:**

Al finalizar este notebook, serás capaz de:
1.  **Combinar múltiples datasets** de forma eficiente utilizando las funciones `concat` y `merge` de pandas.
2.  **Exportar tus DataFrames resultantes** a diversos formatos para su uso posterior.

---

## Importar librerías

In [ ]:
import os
import numpy as np
import pandas as pd

## Mejorar visualización de los dataframes

In [ ]:
# Que muestre todas las columnas
pd.options.display.max_columns = None
# En los dataframes, mostrar los float con dos decimales
pd.options.display.float_format = '{:,.2f}'.format

## Establecer la ruta de los datasets

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

In [ ]:
path = '/content/drive/MyDrive/2026-i-curso-ia-para-economia/datasets'

In [ ]:
# Para establecer el directorio de los archivos
os.chdir(path)

---

## Unir Datos con `pd.concat` y `pd.merge`

Rara vez un proyecto de análisis económico se basa en una sola fuente de datos, por tanto, es muy importante saber combinar datasets de manera lógica y correcta.

### Concatenación con `pd.concat()`

`concat` nos permite "apilar" DataFrames uno encima del otro (eje 0) o uno al lado del otro (eje 1). Su uso más común es para agregar nuevas filas al DataFrame.

**Ejemplo:** Eres un analista del DANE, de la Coordinación de Comercio Internacional, y tienes el archivo de las exportaciones de bienes de julio 2024. Al mes siguiente, recibes el archivo de agosto. `concat` es la acción de tomar las hojas del nuevo reporte y añadirlas al final del reporte del mes anterior. Esto se suele hacer con los archivos de enero a diciembre para construir un único archivo consolidado del año.

In [ ]:
df_expo_jul = pd.read_excel('expo_bienes_2024/07_Exportaciones_2024_Julio.xlsx')
df_expo_jul.head(3)

In [ ]:
df_expo_ago = pd.read_excel('expo_bienes_2024/08_Exportaciones_2024_Agosto.xlsx')
df_expo_ago.head(3)

In [ ]:
df_expo_jul.shape

In [ ]:
df_expo_ago.shape

In [ ]:
# Para concatenar un dataframe debajo del otro (axis=0)
df_expo_2024 = pd.concat([df_expo_jul, df_expo_ago], axis = 0, ignore_index=True)
df_expo_2024.head(3)

In [ ]:
df_expo_2024.tail(3)

In [ ]:
df_expo_2024.shape

In [ ]:
# Para concatenar un dataframe al lado del otro, se coloca axis = 1
df_expo_2024_hor = pd.concat([df_expo_jul, df_expo_ago], axis = 1)
df_expo_2024_hor.head(3)

In [ ]:
df_expo_2024_hor.shape

In [ ]:
df_expo_2024_hor.tail(3)

### Cruce de Información con `pd.merge()`

`merge` es una de las operaciones más poderosas. Emula los `JOIN` de SQL y nos permite combinar DataFrames basándose en el valor de una o más columnas en común (llaves o "keys").

**Ejemplo:** El Banco de la República te contrata. Te entregan una tabla con el Indicador de Seguimiento a la Economía (ISE) para cada mes. Por otro lado, la Bolsa de Valores de Colombia te da una tabla con el volumen de negociación del índice COLCAP, también para cada mes. `merge` es la operación de tomar estas dos tablas y, usando la columna "mes" como el punto de conexión, crear una nueva tabla más rica que contenga tanto el ISE como el volumen del COLCAP en la misma fila para cada mes.

Vamos a explorar los 4 tipos de `merge` más comunes: `inner`, `left`, `right`, y `outer`.

![Tipos de merge](https://drive.google.com/uc?id=1JQ_QH90nibxiBXoZezP4GgtMFNMp7gzV)

In [ ]:
# Para el ejemplo, vamos a crear dos dataframes

# DataFrame 1: Datos de población por país
df_poblacion = pd.DataFrame({
    'codigo_pais': ['COL', 'MEX', 'PER', 'ECU'],
    'poblacion_millones': [52.1, 128.9, 34.0, 18.0]
})

# DataFrame 2: Datos de Llegada de Inversión Extranjera Directa (IED)
# Nota: No tenemos datos para Ecuador (ECU), pero sí para Argentina (ARG), que no está en el otro DF.
df_ied = pd.DataFrame({
    'iso_code': ['COL', 'MEX', 'PER', 'ARG'],
    'ied_millones_usd': [17048, 35292, 11738, 15055]
})

print("Población:")
display(df_poblacion)
print("IED:")
display(df_ied)

#### MERGE TIPO 1: INNER JOIN (Intersección)

Solo nos devuelve las filas donde la llave ('codigo_pais'/'iso_code') existe en AMBOS DataFrames.

In [ ]:
df_inner = pd.merge(df_poblacion, df_ied, left_on='codigo_pais', right_on='iso_code', how='inner')

print("\n--- INNER JOIN ---")
print("Resultado: Solo países presentes en ambos datasets.")
display(df_inner)


#### MERGE TIPO 2: LEFT JOIN

Mantiene TODAS las filas del DataFrame de la izquierda ('df_poblacion') y trae los datos del de la derecha si la llave coincide. Si no, pone NaN.

In [ ]:
df_left = pd.merge(df_poblacion, df_ied, left_on='codigo_pais', right_on='iso_code', how='left')

print("\n--- LEFT JOIN ---")
print("Resultado: Todos los países de 'población', con datos de IED si están disponibles.")
display(df_left)

#### MERGE TIPO 3: RIGHT JOIN
Mantiene TODAS las filas del DataFrame de la derecha ('df_ied') y trae los datos del de la izquierda si la llave coincide.

In [ ]:
df_right = pd.merge(df_poblacion, df_ied, left_on='codigo_pais', right_on='iso_code', how='right')

print("\n--- RIGHT JOIN ---")
print("Resultado: Todos los países de 'IED', con datos de población si están disponibles.")
display(df_right)

#### MERGE TIPO 4: OUTER JOIN (Unión Completa)

Mantiene TODAS las filas de AMBOS DataFrames. Rellena con NaN donde no hay correspondencia.

In [ ]:
df_outer = pd.merge(df_poblacion, df_ied, left_on='codigo_pais', right_on='iso_code', how='outer')

print("\n--- OUTER JOIN ---")
print("Resultado: Todos los países de ambos datasets.")
display(df_outer)

#### Ejercicios

1. Tienes el siguiente dataframe `df_econ` con datos económicos para Colombia, México, Perú y Brasil y el dataframe `df_poblacion` que creamos previamente. Usando un `merge`, crea un nuevo DataFrame llamado `df_final` que contenga el PIB per cápita, la inflación, las exportaciones y la población para cada país. ¿Qué tipo de `join` es el más apropiado si quieres mantener todos los datos de `df_econ`?

In [ ]:
# Datos económicos por país
df_econ = pd.DataFrame({
    'pais_iso': ['COL', 'MEX', 'PER', 'BRA'],
    'PIB_per_capita': [7917, 13954, 8316, 10296],
    'Inflacion': [6.61, 4.72, 2.01, 4.83],
    'Exportaciones': [49600, 680790, 74700, 337000]
})
df_econ

In [ ]:
# Respuesta:


In [ ]:
# La columna codigo_pais sobra, por lo que la vamos a borrar


2. Tenemos estas dos tablas, una con países y su PIB, y otra con países y su índice de desarrollo humano (IDH). No todos los países están en ambas tablas. Discutan con su compañero qué tabla resultante obtendrían con un inner join vs un outer join.

In [ ]:
# DataFrame 1: Datos de PIB por país
df_pib = pd.DataFrame({
    'pais': ['BRA', 'ARG', 'PER', 'CHL'],
    'pib_millones_usd': [2188, 604, 283, 329]
})

# DataFrame 2: Datos IDH por país
df_idh = pd.DataFrame({
    'pais': ['BRA', 'CHL', 'COL', 'PER', 'MEX'],
    'IDH': [0.786, 0.878, 0.788, 0.794, 0.789],
    'Categoría': ['Alto', 'Muy alto', 'Alto', 'Alto', 'Alto']
})

print("PIB:")
display(df_pib)
print("IDH:")
display(df_idh)

In [ ]:
# Inner join

In [ ]:
# Outer join


3.  He intentado hacer un merge entre estos dos DataFrames, pero Python me da un error. ¿Quién puede ver por qué está fallando el código?

In [ ]:
# DataFrame 1: Datos de PIB per cápita por país
df_pib_pc = pd.DataFrame({
    'país_iso': ['COL', 'MEX', 'PER', 'BRA'],
    'PIB_per_capita': [7917, 13954, 8316, 10296]
})

# DataFrame 2: Datos de inflación por país
df_inflacion = pd.DataFrame({
    'pais_iso': ['COL', 'MEX', 'PER', 'BRA'],
    'Inflacion': [6.61, 4.72, 2.01, 4.83]
})

print("PIB per cápita:")
display(df_pib_pc)
print("Inflación:")
display(df_inflacion)

In [ ]:
df_merge = pd.merge(df_pib_pc, df_inflacion, on='pais_iso', how='inner')
df_merge

In [ ]:
# Solución


4. Y en este merge, ¿qué está fallando?

In [ ]:
# DataFrame 1: Datos de PIB en USD por municipio
df_pib_usd = pd.DataFrame({
    'cod_municipio': ['85424', '16784', '13968', '15978'],
    'PIB_usd': [417, 1848, 283, 2188]
})

# DataFrame 2: Datos de poblacion por municipio
df_poblacion = pd.DataFrame({
    'cod_municipio': [85424, 16784, 13968, 15978],
    'Población': ['49600', '680790', '74700', '337000']
})

print("PIB en USD:")
display(df_pib_usd)
print("Población:")
display(df_poblacion)

In [ ]:
df_combinado = pd.merge(df_pib_usd, df_poblacion, on='cod_municipio', how='inner')
df_combinado

In [ ]:
# Solución


#### Pregunta de discusión

¿Qué harían si al intentar unir dos tablas del gobierno por "nombre del municipio", descubren que "Bogotá D.C." en una tabla se llama "Bogota" y en la otra se llama Bogotá? ¿Cómo afecta esto a los diferentes tipos de join?

#### Ejemplos aplicados a Economía

- **Scoring de Crédito:** Unir el historial de crédito de un cliente (desde una base de datos SQL) con sus datos de redes sociales (desde una API que devuelve JSON) para crear un perfil de riesgo más completo.

- **Análisis de Política Pública:** Combinar datos de ejecución presupuestal de municipios (archivo CSV del gobierno) con indicadores de pobreza del DANE (tabla en Excel) con el fin de evaluar el impacto del gasto.

- **Investigación Macroeconómica:** Fusionar series de tiempo 2000-2025 sobre la inflación y el desempleo de todos los países de Latam (varios archivos de Excel) con el fin de crear un panel de datos y analizar el comportamiento de estas dos variables en el tiempo.

---

## Guardar / exportar datos

Una vez que hemos limpiado, combinado y analizado nuestros datos, el último paso es guardarlos. Esto nos permite compartir resultados o usarlos en otros programas (como R, Stata, o Tableau).

### Guardar en CSV (Comma-Separated Values) - el más común

In [ ]:
df_combinado.to_csv('Descargas/Datos_municipio.csv', encoding ='utf-8', index = False)

### Guardar en TXT

In [ ]:
df_combinado.to_csv('Descargas/Datos_municipio.txt', sep="|", decimal=',', encoding ='utf-8', index= False)

### Guardar en JSON

El formato 'records' es muy útil para la web y APIs. Significa que cada fila del DataFrame se convertirá en un diccionario (JSON object) dentro de una lista.

indent=4: Agrega sangría de 4 espacios en el JSON para que sea más legible.
Si no se coloca, todo el JSON queda en una sola línea.

In [ ]:
df_final.to_json('Descargas/datos_macro_final.json', orient='records', indent=4)

### Guardar en Excel

In [ ]:
df_combinado.to_excel("Descargas/Datos_municipio.xlsx", index=False)

#### Guardar más de un dataframe en Excel

In [ ]:
with pd.ExcelWriter('Descargas/Ejemplo.xlsx') as writer:
    df_combinado.to_excel(writer, sheet_name='Municipio', index=False)
    df_final.to_excel(writer, sheet_name='Macro', index=False)

#### Guardar un dataframe en un Excel que ya existe como una hoja adicional (mode='a)

Si la hoja ya existe, la reemplaza.

In [ ]:
with pd.ExcelWriter('Descargas/Ejemplo.xlsx', mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
    df_expo_2024.to_excel(writer, sheet_name='Expo', index = False)